In [ ]:
import tkinter as tk
import random
import heapq
import time
import math

ROWS = 20
COLS = 20
CELL = 25
SPAWN_PROB = 0.08

WIDTH = COLS * CELL
HEIGHT = ROWS * CELL

counter = 0
mode = "astar"
heuristic_mode = "manhattan"

last_expanded = 0
last_cost = 0
last_time = 0

class Node:
    def __init__(self, r, c):
        self.r = r
        self.c = c
        self.wall = False
        self.parent = None
        self.visited = False
        self.frontier = False

def make_grid():
    return [[Node(r, c) for c in range(COLS)] for r in range(ROWS)]

grid = make_grid()

start = grid[0][0]
goal = grid[ROWS-1][COLS-1]
agent_pos = start

def manhattan(a, b):
    return abs(a.r - b.r) + abs(a.c - b.c)

def euclidean(a, b):
    return math.sqrt((a.r - b.r)**2 + (a.c - b.c)**2)

def h(a, b):
    if heuristic_mode == "manhattan":
        return manhattan(a, b)
    else:
        return euclidean(a, b)

def neighbors(n):
    dirs = [(1,0),(-1,0),(0,1),(0,-1)]
    result = []
    for dr, dc in dirs:
        r = n.r + dr
        c = n.c + dc
        if 0 <= r < ROWS and 0 <= c < COLS:
            if not grid[r][c].wall:
                result.append(grid[r][c])
    return result

def reset_nodes():
    for row in grid:
        for n in row:
            n.parent = None
            n.visited = False
            n.frontier = False

def build_path(end):
    path = []
    while end.parent:
        path.append(end)
        end = end.parent
    path.reverse()
    return path

def astar():
    global counter
    reset_nodes()
    openq = []
    g = {agent_pos: 0}

    counter += 1
    heapq.heappush(openq, (0, counter, agent_pos))

    expanded = 0
    t0 = time.time()

    while openq:
        _, _, cur = heapq.heappop(openq)
        if cur.visited:
            continue

        cur.visited = True
        expanded += 1

        if cur == goal:
            return build_path(goal), expanded, (time.time()-t0)*1000

        for nb in neighbors(cur):
            newg = g[cur] + 1
            if nb not in g or newg < g[nb]:
                g[nb] = newg
                f = newg + h(nb, goal)
                nb.parent = cur
                nb.frontier = True

                counter += 1
                heapq.heappush(openq, (f, counter, nb))

    return [], expanded, 0

def greedy():
    global counter
    reset_nodes()
    openq = []
    visited = set()

    counter += 1
    heapq.heappush(openq, (h(agent_pos, goal), counter, agent_pos))

    expanded = 0
    t0 = time.time()

    while openq:
        _, _, cur = heapq.heappop(openq)
        if cur in visited:
            continue

        visited.add(cur)
        cur.visited = True
        expanded += 1

        if cur == goal:
            return build_path(goal), expanded, (time.time()-t0)*1000

        for nb in neighbors(cur):
            if nb not in visited:
                nb.parent = cur
                nb.frontier = True

                counter += 1
                heapq.heappush(openq, (h(nb, goal), counter, nb))

    return [], expanded, 0

def random_map():
    global agent_pos, running

    agent_pos = start
    running = False

    try:
        density = float(density_entry.get())
    except:
        density = 0.3

    for row in grid:
        for n in row:
            if n not in (start, goal):
                n.wall = random.random() < density

    reset_nodes()
    draw()

def draw(path=None, expanded=0, cost=0, ms=0):
    canvas.delete("all")

    for row in grid:
        for n in row:
            x1 = n.c * CELL
            y1 = n.r * CELL
            x2 = x1 + CELL
            y2 = y1 + CELL

            color = "white"
            if n.wall:
                color = "black"
            elif n.visited:
                color = "lightblue"
            elif n.frontier:
                color = "yellow"

            canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="gray")

    if path:
        for p in path:
            canvas.create_rectangle(
                p.c*CELL, p.r*CELL,
                p.c*CELL+CELL, p.r*CELL+CELL,
                fill="green"
            )

    canvas.create_rectangle(agent_pos.c*CELL, agent_pos.r*CELL,
                            agent_pos.c*CELL+CELL, agent_pos.r*CELL+CELL,
                            fill="blue")

    canvas.create_rectangle(goal.c*CELL, goal.r*CELL,
                            goal.c*CELL+CELL, goal.r*CELL+CELL,
                            fill="red")

    info.config(text=f"Nodes:{expanded}  Cost:{cost}  Time:{int(ms)}ms")

def click(event):
    c = event.x // CELL
    r = event.y // CELL
    node = grid[r][c]
    if node not in (agent_pos, goal):
        node.wall = not node.wall
    draw()

def set_mode(m):
    global mode
    mode = m

def set_heuristic(m):
    global heuristic_mode
    heuristic_mode = m

current_path = []
agent_index = 0
running = False

def run_search():
    global current_path, agent_index, running
    global last_expanded, last_cost, last_time

    if mode == "astar":
        path, exp, ms = astar()
    else:
        path, exp, ms = greedy()

    current_path = path
    agent_index = 0
    running = True

    last_expanded = exp
    last_cost = len(path)
    last_time = ms

    draw(path, last_expanded, last_cost, last_time)
    move_agent()

def move_agent():
    global agent_index, running, agent_pos

    if not running:
        return

    if agent_index >= len(current_path):
        running = False
        return

    next_node = current_path[agent_index]

    if next_node.wall:
        run_search()
        return

    agent_pos = next_node
    agent_index += 1
    draw(current_path, last_expanded, last_cost, last_time)

    root.after(200, move_agent)

dynamic_on = False

def toggle_dynamic():
    global dynamic_on
    dynamic_on = not dynamic_on
    dynamic_button.config(text="Dynamic: ON" if dynamic_on else "Dynamic: OFF")

def dynamic_step():
    if dynamic_on and running:
        if random.random() < SPAWN_PROB:
            r = random.randint(0, ROWS-1)
            c = random.randint(0, COLS-1)
            node = grid[r][c]
            if node not in (agent_pos, goal):
                node.wall = True
                draw(current_path, last_expanded, last_cost, last_time)

    root.after(300, dynamic_step)

root = tk.Tk()
root.title("Dynamic Pathfinding Agent")

canvas = tk.Canvas(root, width=WIDTH, height=HEIGHT)
canvas.pack()
canvas.bind("<Button-1>", click)

frame = tk.Frame(root)
frame.pack()

info = tk.Label(frame, text="Ready")
info.pack()

tk.Button(frame, text="A*", command=lambda: set_mode("astar")).pack(side="left")
tk.Button(frame, text="Greedy", command=lambda: set_mode("greedy")).pack(side="left")

tk.Button(frame, text="Manhattan",
          command=lambda: set_heuristic("manhattan")).pack(side="left")

tk.Button(frame, text="Euclidean",
          command=lambda: set_heuristic("euclidean")).pack(side="left")

tk.Button(frame, text="Run", command=run_search).pack(side="left")

density_entry = tk.Entry(frame, width=5)
density_entry.insert(0, "0.2")
density_entry.pack(side="left")

tk.Button(frame, text="Random Map", command=random_map).pack(side="left")

dynamic_button = tk.Button(frame, text="Dynamic: OFF",
                           command=toggle_dynamic)
dynamic_button.pack(side="left")

draw()
dynamic_step()
root.mainloop()